# 00 — Setup check

What this pod is configured to run, before anything expensive happens.
Needs no stage outputs — safe to run on a fresh pod.

Run top to bottom. Nothing here writes to `outputs/`.

In [ ]:
import sys, pathlib

REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(REPO))

import config  # sets HF_HOME — must come before any transformers import

print(config.summary())

## Hardware

The authoritative check is `python preflight.py` — this is the same numbers, inline.

In [ ]:
import shutil
import torch
import pandas as pd

GB = 1024**3
rows = []
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    free_b, total_b = torch.cuda.mem_get_info(i)
    rows.append({
        "device": i,
        "name": p.name,
        "arch": f"sm_{p.major}{p.minor}",
        "vram_total_gb": round(total_b / GB, 1),
        "vram_free_gb": round(free_b / GB, 1),
    })

print(f"torch {torch.__version__}  cuda {torch.version.cuda}  available={torch.cuda.is_available()}")
print(f"{config.PROBED_MODEL} in {config.DTYPE} needs ~{config.APPROX_WEIGHTS_GB}GB\n")
display(pd.DataFrame(rows))

paths = {"HF_HOME": config.HF_HOME, "repo": config.REPO_ROOT, "/dev/shm": pathlib.Path("/dev/shm")}
display(pd.DataFrame([
    {"path": str(v), "label": k,
     "free_gb": round(shutil.disk_usage(v).free / GB, 1),
     "total_gb": round(shutil.disk_usage(v).total / GB, 1)}
    for k, v in paths.items() if v.exists()
]))

## Personas

Ordered by *intuited* distance from the default assistant. Stage 6 measures the
real thing and is allowed to disagree — that comparison is one of the results.

`[PLACEHOLDER]` means the system prompt is still scaffold text.

In [ ]:
import personas

def _state(p):
    if p["system_prompt"] is None:
        return "no system message (template default)"
    return "PLACEHOLDER" if p["system_prompt"].startswith("TODO") else "written"

df = pd.DataFrame([
    {"rank": p["distance_rank"], "id": p["id"], "name": p["name"],
     "prompt": _state(p), "chars": len(p["system_prompt"] or "")}
    for p in personas.PERSONAS
]).set_index("rank")

todo = (df["prompt"] == "PLACEHOLDER").sum()
print(f"axis endpoints: {personas.DEFAULT_PERSONA_ID} -> {personas.MOST_DISTANT_PERSONA_ID}")
print(f"{todo} of {len(df)} system prompts still placeholder\n")
display(df)

## Topics and the experiment grid

In [ ]:
import scenarios

topics, source = scenarios.load_topics()
print(f"\nsource={source}, {len(topics)} topics. First 10:")
for i, t in enumerate(topics[:10]):
    print(f"  {i:3d}  {t}")
if source == "fallback":
    print("\nWARNING: hardcoded fallback list, not the published dataset.")

In [ ]:
n_calls = len(personas.PERSONAS) * len(config.EMOTIONS) * config.N_SCENARIOS_PER_CELL
est_out_tok = n_calls * config.GEN_MAX_TOKENS
est_in_tok = n_calls * 250  # rough: system prompt + instruction
cost = (est_in_tok / 1e6 * config.GENERATOR_PRICE_IN_PER_MTOK
        + est_out_tok / 1e6 * config.GENERATOR_PRICE_OUT_PER_MTOK)

print(f"{len(personas.PERSONAS)} personas x {len(config.EMOTIONS)} emotions "
      f"x {config.N_SCENARIOS_PER_CELL} scenarios = {n_calls} generations")
print(f"upper-bound cost estimate: ${cost:.2f} at list price for {config.GENERATOR_MODEL}")
print(f"(assumes every call maxes out at {config.GEN_MAX_TOKENS} output tokens — real cost will be lower)")

---
**Next:** `python preflight.py` for the authoritative blocker check, then
`python gen_data.py --limit 4` to smoke test the generation path.
Results land in `01_data.ipynb` and `02_results.ipynb`.